# Pydantic: structured data for agents

Companion notebook for **lesson 4.3**. Everything up to section 6 runs without a model: define a model, validate and parse, put its schema in a prompt, the two gotchas, and the Order exercise. Section 6 uses a real model to extract structured data, validated by the same Pydantic model, with one retry on failure.

**Run it in Colab, setup cell first.** It uses Colab's built-in models (no API key); uncomment a `MODEL = ...` line there to choose one.

In [ ]:
# Setup: Colab's built-in models, no API key needed. Run this cell first.
from google.colab import ai
print("Available models:", ", ".join(ai.list_models()))

MODEL = None                              # Colab's default; uncomment a line below to choose
# MODEL = "google/gemini-2.5-flash"
# MODEL = "google/gemini-2.5-pro"
# MODEL = "google/gemini-3.8-flash"

def get_completion(messages):
    """Send chat messages to the model. Colab's generate_text takes one prompt, so roles are inlined."""
    prompt = "\n\n".join(f"{m['role'].upper()}:\n{m['content']}" for m in messages) + "\n\nASSISTANT:\n"
    return ai.generate_text(prompt, model_name=MODEL) if MODEL else ai.generate_text(prompt)

## 1 · A model is a schema

In [1]:
import json
from typing import List, Optional
from pydantic import BaseModel, ConfigDict, Field, ValidationError

class UserProfile(BaseModel):
    """Structured information about a user."""
    user_id: int = Field(..., description="The unique identifier for the user.")
    username: str = Field(..., description="The user's public username.")
    is_active: bool = Field(..., description="Whether the account is currently active.")
    email: Optional[str] = Field(None, description="The user's email address, if available.")

def problems(err: ValidationError) -> str:
    """One line per problem: 'field: message'."""
    return "\n".join(f"  {'.'.join(map(str, e['loc'])) or '(root)'}: {e['msg']}" for e in err.errors())

## 2 · Validate and parse

In [2]:
profile = UserProfile.model_validate_json('{"user_id": 42, "username": "alex", "is_active": true}')
print("parsed:", profile)
print("dot access:", profile.username, "| missing optional field:", profile.email)

for raw in ['{"user_id": "abc", "username": "alex"}',          # wrong type + missing field
            'Sure! Here is the profile: {"user_id": 42}']:      # chatty, not JSON
    try:
        UserProfile.model_validate_json(raw)
    except ValidationError as e:
        print(f"\nrejected {raw[:40]!r}\n{problems(e)}")

parsed: user_id=42 username='alex' is_active=True email=None
dot access: alex | missing optional field: None

rejected '{"user_id": "abc", "username": "alex"}'
  user_id: Input should be a valid integer, unable to parse string as an integer
  is_active: Field required

rejected 'Sure! Here is the profile: {"user_id": 4'
  (root): Invalid JSON: expected value at line 1 column 1


## 3 · The same model, as a schema for the prompt

In [3]:
schema = UserProfile.model_json_schema()
print(json.dumps(schema, indent=2))

{
  "description": "Structured information about a user.",
  "properties": {
    "user_id": {
      "description": "The unique identifier for the user.",
      "title": "User Id",
      "type": "integer"
    },
    "username": {
      "description": "The user's public username.",
      "title": "Username",
      "type": "string"
    },
    "is_active": {
      "description": "Whether the account is currently active.",
      "title": "Is Active",
      "type": "boolean"
    },
    "email": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "description": "The user's email address, if available.",
      "title": "Email"
    }
  },
  "required": [
    "user_id",
    "username",
    "is_active"
  ],
  "title": "UserProfile",
  "type": "object"
}


## 4 · Two gotchas

**Optional ≠ can be left out:** `Optional[...]` decides which values are allowed; a default decides whether the key may be missing. **Lax mode converts types** unless you ask for strict.

In [4]:
class A(BaseModel): email: str
class B(BaseModel): email: Optional[str]
class C(BaseModel): email: Optional[str] = None

def outcome(model, data):
    try:
        return repr(model.model_validate(data).email)
    except ValidationError as e:
        return "✗ " + e.errors()[0]["msg"]

print(f"{'declaration':29} {'key missing':26} null")
for label, model in [("email: str", A), ("email: Optional[str]", B), ("email: Optional[str] = None", C)]:
    print(f"{label:29} {outcome(model, {}):26} {outcome(model, {'email': None})}")

class Lax(BaseModel):
    user_id: int
    is_active: bool

class Strict(Lax):
    model_config = ConfigDict(strict=True)

raw = '{"user_id": "42", "is_active": "yes"}'
print("\nlax:   ", Lax.model_validate_json(raw))
try:
    Strict.model_validate_json(raw)
except ValidationError as e:
    print("strict:\n" + problems(e))

declaration                   key missing                null
email: str                    ✗ Field required           ✗ Input should be a valid string
email: Optional[str]          ✗ Field required           None
email: Optional[str] = None   None                       None

lax:    user_id=42 is_active=True
strict:
  user_id: Input should be a valid integer
  is_active: Input should be a valid boolean


## 5 · Exercise: an Order with nested items

`OrderItem` must come first, `List[OrderItem]` validates every item, `min_length=1` enforces "one or more", and optional fields need a default.

In [5]:
class OrderItem(BaseModel):
    sku: str
    quantity: int = Field(..., gt=0)
    item_name: Optional[str] = None

class Order(BaseModel):
    order_id: int
    customer_email: Optional[str] = None
    items: List[OrderItem] = Field(..., min_length=1)
    total_amount: float = Field(..., ge=0)

print(Order.model_validate({"order_id": 1001, "items": [{"sku": "MUG-01", "quantity": 2}], "total_amount": 24.5}))

for bad in [{"order_id": 1001, "items": [{"sku": "MUG-01", "quantity": "two"}], "total_amount": 24.5},
            {"order_id": 1001, "items": [], "total_amount": 0},
            {"order_id": 1001, "items": [{"sku": "MUG-01", "quantity": 0}], "total_amount": 0}]:
    try:
        Order.model_validate(bad)
    except ValidationError as e:
        print("\nrejected:\n" + problems(e))

order_id=1001 customer_email=None items=[OrderItem(sku='MUG-01', quantity=2, item_name=None)] total_amount=24.5

rejected:
  items.0.quantity: Input should be a valid integer, unable to parse string as an integer

rejected:
  items: List should have at least 1 item after validation, not 0

rejected:
  items.0.quantity: Input should be greater than 0


## 6 · Extract with a real model, validated by the same model

The schema goes into the prompt; the reply is validated; on failure the errors go back once. The extraction function runs anywhere; the last cell calls the model.

In [6]:
import re

def extract(text: str, model: type[BaseModel] = UserProfile, retries: int = 1):
    prompt = (f"Extract the data from the text below. Respond with only a JSON object matching this "
              f"JSON schema:\n{json.dumps(model.model_json_schema())}\n\nText:\n{text}")
    messages = [{"role": "user", "content": prompt}]
    for attempt in range(retries + 1):
        reply = get_completion(messages)
        raw = re.sub(r"^\s*`{3}(?:json)?\s*|\s*`{3}\s*$", "", reply.strip())   # strip a Markdown fence
        try:
            return model.model_validate_json(raw)
        except ValidationError as e:
            print(f"attempt {attempt + 1} rejected:\n{problems(e)}")
            messages += [{"role": "assistant", "content": reply},
                         {"role": "user", "content": f"That failed validation:\n{problems(e)}\nReturn corrected JSON only."}]
    return None

In [ ]:
print(extract("Hi, I'm Alex (user #42). My account is active; reach me at alex@example.com."))
print(extract("Order 1001: two MUG-01 mugs and one 'Blue cap' (sku CAP-7) for 31.40 total.", model=Order))